## RoBERTa model: fine-tuned for binary sequence classification on the SRAQ dataset

In [3]:
# imports

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch
from torch import nn
import pandas as pd
from transformers import AutoTokenizer
from datasets import Dataset
from transformers import DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, confusion_matrix

/home/gusbeihel@GU.GU.SE/ml2-final-project/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# loading processed data

train = pd.read_json("../data/train_processed.jsonl", lines=True)
valid = pd.read_json("../data/dev_processed.jsonl", lines=True)
test = pd.read_json("../data/test_processed.jsonl", lines=True)

### Tokenization

In [5]:
# loading tokenizer

tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

# looking at an example
question = train["question"].iloc[0]
tokens = tokenizer.tokenize(question)
print(tokens)

['Are', 'Ġwe', 'Ġentitled', ',', 'Ġfor', 'Ġexample', ',', 'Ġto', 'Ġrequire', 'Ġthat', 'Ġnew', 'Ġimmigrants', 'Ġbe', 'Ġof', 'Ġa', 'Ġcertain', 'Ġethnic', 'Ġor', 'Ġreligious', 'Ġbackground', ',', 'Ġare', 'Ġwe', 'Ġentitled', 'Ġto', 'Ġrequire', 'Ġthat', 'Ġthey', 'Ġpay', 'Ġextremely', 'Ġhigh', 'Ġfees', 'Ġin', 'Ġreturn', 'Ġfor', 'Ġcitizenship', ',', 'Ġare', 'Ġwe', 'Ġallowed', 'Ġattach', 'Ġconditions', 'Ġwhich', 'Ġhold', 'Ġeven', 'Ġafter', 'Ġcitizenship', 'Ġhas', 'Ġbeen', 'Ġgranted', ',', 'Ġand', 'Ġif', 'Ġnot', ',', 'Ġwhy', 'Ġnot', '?']


In [6]:
# mapping "rhetorical" and "informational" labels to integers

label2id = {"informational": 0, "rhetorical": 1}
id2label = {0: "informational", 1: "rhetorical"}

# creating new column with integers
train["label_int"] = train["label"].map(label2id)

valid["label_int"] = valid["label"].map(label2id)

test["label_int"] = test["label"].map(label2id)

In [7]:
# truncating text that exceeds the maximum length

# truncating from the left (beginning)
tokenizer.truncation_side = "left"

def preprocess(context, question):
    return tokenizer(
        context,
        question,
        truncation="only_first", # truncates only context (from the left)
        max_length=512
    )

I truncated the text from the left to preserve the context closest to the question.

In [8]:
# tokenizing training data

train_tokenized = train.apply(
    lambda row: preprocess(row["preceding_turns"], row["question"]),
    axis=1
)

# checking
print(train_tokenized.iloc[0].keys())
print(len(train_tokenized.iloc[0]["input_ids"]))

# converting tokenized data into dataframe
train_encoded = pd.DataFrame(train_tokenized.tolist())

# adding integer labels
train_encoded["labels"] = train["label_int"].to_numpy()

train_encoded.head()

KeysView({'input_ids': [0, 74, 202, 28, 10, 538, 3855, 6, 25, 171, 3354, 3458, 82, 74, 45, 28, 2460, 7, 185, 215, 41, 14270, 4, 38, 64, 45, 4744, 10, 36906, 3354, 3346, 145, 2882, 7, 342, 39, 865, 15, 5, 42244, 8, 24909, 37, 2046, 11, 383, 14, 32, 11, 15441, 20874, 7, 39812, 488, 4, 152, 74, 1266, 24, 74, 33, 10, 1233, 10142, 913, 11, 4881, 5, 936, 7510, 566, 92, 4175, 6, 150, 45, 145, 10, 7926, 7, 7212, 453, 9, 70, 3458, 34332, 4, 50118, 50118, 24788, 35, 598, 2519, 7, 5, 1049, 8823, 38, 524, 562, 35, 50118, 50118, 1640, 134, 43, 22, 1213, 115, 95, 6105, 111, 99, 18, 5, 477, 845, 1009, 100, 33, 460, 2334, 7, 42, 11, 127, 1461, 618, 4, 286, 1840, 18, 12253, 6, 2073, 5, 16490, 618, 137, 6827, 8, 21025, 961, 1493, 18, 86, 44460, 287, 2801, 6, 150, 171, 74, 28, 2882, 7, 6105, 6, 5, 38004, 7958, 74, 533, 45, 28, 11, 41, 14270, 7, 1840, 4, 10623, 52, 74, 1888, 5, 346, 9, 38004, 12, 25331, 23907, 16281, 6, 8, 33, 5, 2194, 14547, 31, 5, 786, 12, 5982, 1023, 9997, 7958, 16281, 4, 280, 16, 41, 

,input_ids,attention_mask,labels
0,"[0, 74, 202, 28, 10, 538, 3855, 6, 25, 171, 33...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
1,"[0, 41761, 1215, 250, 35, 20560, 203, 99, 24, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
2,"[0, 41761, 1215, 250, 35, 178, 141, 171, 82, 7...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
3,"[0, 41761, 1215, 250, 35, 20, 132, 207, 9061, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
4,"[0, 41761, 1215, 250, 35, 345, 16, 117, 3050, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0


In [9]:
# converting to Hugging Face dataset

train_dataset = Dataset.from_pandas(
    train_encoded,
    preserve_index=False
)

print(train_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 345
})


In [10]:
# tokenizing validation data

valid_tokenized = valid.apply(
    lambda row: preprocess(row["preceding_turns"], row["question"]),
    axis=1
)

# checking
print(valid_tokenized.iloc[0].keys())
print(len(valid_tokenized.iloc[0]["input_ids"]))

# converting tokenized data into dataframe
valid_encoded = pd.DataFrame(valid_tokenized.tolist())

# adding integer labels
valid_encoded["labels"] = valid["label_int"].to_numpy()

valid_encoded.head()

KeysView({'input_ids': [0, 41761, 1215, 250, 35, 38, 437, 11, 5, 609, 9, 2159, 10, 20568, 8, 65, 9, 127, 934, 1379, 16, 127, 19960, 18, 10029, 4, 1437, 264, 855, 34, 155, 3678, 8, 132, 10017, 4, 1437, 264, 18, 10, 1307, 3477, 16095, 8, 10, 9087, 2903, 98, 79, 14534, 13, 106, 157, 8, 51, 214, 157, 30758, 36, 26837, 5, 6185, 4758, 6, 38, 24909, 37, 18, 5, 22406, 322, 50118, 50118, 11913, 5924, 6, 144, 9, 5, 2127, 38, 437, 546, 7, 907, 34, 10, 155, 4716, 714, 3000, 4, 1437, 38, 437, 45, 3915, 59, 65, 9, 5, 10017, 142, 24, 18, 41, 11894, 4758, 8, 5907, 40, 655, 192, 123, 4, 50118, 50118, 1213, 214, 70, 157, 30758, 8, 218, 75, 28847, 98, 38, 218, 75, 206, 24, 581, 28, 10, 380, 936, 6, 53, 127, 129, 2490, 16, 114, 38, 362, 5, 3678, 15, 5792, 23, 363, 82, 74, 192, 155, 8, 77, 38, 213, 66, 7, 120, 5, 6185, 4758, 23, 363, 51, 581, 216, 38, 33, 23, 513, 204, 4, 1437, 38, 348, 16425, 11, 34003, 137, 8, 216, 89, 32, 460, 6611, 54, 32, 23809, 8, 2882, 7, 11316, 7, 5, 289, 16024, 59, 932, 4, 50118, 

,input_ids,attention_mask,labels
0,"[0, 41761, 1215, 250, 35, 38, 437, 11, 5, 609,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
1,"[0, 41761, 1215, 250, 35, 646, 5593, 12677, 74...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
2,"[0, 41761, 1215, 250, 35, 280, 2682, 16, 4968,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
3,"[0, 41761, 1215, 250, 35, 359, 19377, 131, 100...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
4,"[0, 41761, 1215, 250, 35, 38, 206, 14, 31682, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1


In [11]:
# converting to Hugging Face dataset

valid_dataset = Dataset.from_pandas(
    valid_encoded,
    preserve_index=False
)

print(valid_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 90
})


In [12]:
# tokenizing test data: real context
test_tokenized_real = test.apply(
    lambda row: preprocess(row["preceding_turns"], row["question"]),
    axis=1
)

# checking
print(test_tokenized_real.iloc[0].keys())
print(len(test_tokenized_real.iloc[0]["input_ids"]))

# converting tokenized data into dataframe
test_encoded_real = pd.DataFrame(test_tokenized_real.tolist())

# adding integer labels
test_encoded_real["labels"] = test["label_int"].to_numpy()

test_encoded_real.head()

KeysView({'input_ids': [0, 41761, 1215, 250, 35, 85, 18, 31110, 260, 1344, 5971, 6, 9, 768, 5, 2788, 74, 28, 59, 12933, 6, 97, 749, 32, 45, 2034, 15, 5, 34610, 4, 50118, 41761, 1215, 10975, 2794, 36140, 42645, 5902, 95, 584, 99, 5, 6097, 4289, 74, 28, 77, 1782, 6821, 35472, 50118, 41761, 1215, 387, 35, 939, 3640, 2980, 24, 4, 2, 2, 9178, 98, 116, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]})
66


,input_ids,attention_mask,labels
0,"[0, 41761, 1215, 250, 35, 85, 18, 31110, 260, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
1,"[0, 173, 13, 6162, 6, 258, 11, 645, 7, 582, 16...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
2,"[0, 8, 49, 4554, 16, 10649, 13709, 14079, 99, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
3,"[0, 16, 10, 41887, 31, 110, 42976, 4, 166, 101...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
4,"[0, 28241, 249, 4, 50118, 41761, 1215, 387, 35...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1


In [13]:
# tokenizing test data: swapped context
test_tokenized_swapped = test.apply(
    lambda row: preprocess(row["swapped_context"], row["question"]),
    axis=1
)

# checking
print(test_tokenized_swapped.iloc[0].keys())
print(len(test_tokenized_swapped.iloc[0]["input_ids"]))

# converting tokenized data into dataframe
test_encoded_swapped = pd.DataFrame(test_tokenized_swapped.tolist())

# adding integer labels
test_encoded_swapped["labels"] = test["label_int"].to_numpy()

test_encoded_swapped.head()

KeysView({'input_ids': [0, 41761, 1215, 250, 35, 280, 18, 45, 269, 1528, 4, 16210, 6265, 16, 5, 169, 7, 519, 2099, 19, 10, 693, 11, 937, 4, 50118, 41761, 1215, 387, 35, 359, 19377, 131, 13540, 43253, 12606, 6265, 16, 5, 169, 7, 519, 2099, 19, 10, 693, 11, 937, 4, 50118, 50118, 5096, 19266, 47, 240, 69, 7, 2854, 7, 1533, 430, 383, 6, 3099, 69, 8394, 7, 213, 15, 10, 1248, 19, 47, 965, 75, 7132, 154, 7, 33, 2099, 19, 47, 4, 1437, 50118, 50118, 35490, 1262, 965, 75, 14176, 7, 33, 1363, 31677, 50, 932, 1493, 4, 50118, 41761, 1215, 250, 35, 359, 19377, 131, 35490, 1262, 965, 75, 14176, 7, 33, 1363, 31677, 50, 932, 1493, 4, 50118, 50118, 243, 115, 28, 1687, 10, 1149, 1567, 24, 734, 2, 2, 9178, 98, 116, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

,input_ids,attention_mask,labels
0,"[0, 41761, 1215, 250, 35, 280, 18, 45, 269, 15...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
1,"[0, 47, 192, 10, 1129, 14, 34, 3187, 65, 6, 24...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
2,"[0, 555, 10, 3299, 114, 10, 11671, 313, 1516, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
3,"[0, 5709, 6, 1522, 5938, 13, 390, 32, 4446, 8,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
4,"[0, 504, 107, 16, 33370, 459, 4779, 4378, 23, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1


In [14]:
# tokenizing test data: empty context
test_tokenized_empty = test.apply(
    lambda row: preprocess("", row["question"]),
    axis=1
)

# checking
print(test_tokenized_empty.iloc[0].keys())
print(len(test_tokenized_empty.iloc[0]["input_ids"]))

# converting tokenized data into dataframe
test_encoded_empty = pd.DataFrame(test_tokenized_empty.tolist())

# adding integer labels
test_encoded_empty["labels"] = test["label_int"].to_numpy()

test_encoded_empty.head()

KeysView({'input_ids': [0, 2, 2, 9178, 98, 116, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]})
7


,input_ids,attention_mask,labels
0,"[0, 2, 2, 9178, 98, 116, 2]","[1, 1, 1, 1, 1, 1, 1]",0
1,"[0, 2, 2, 42847, 67, 356, 62, 383, 36, 179, 76...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
2,"[0, 2, 2, 2847, 116, 2]","[1, 1, 1, 1, 1, 1]",1
3,"[0, 2, 2, 8275, 5, 249, 95, 14529, 106, 16340,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1
4,"[0, 2, 2, 32251, 817, 47, 430, 87, 470, 2378, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1


In [15]:
# converting to Hugging Face dataset

test_dataset_real = Dataset.from_pandas(
    test_encoded_real,
    preserve_index=False
)
print(test_dataset_real)

test_dataset_swapped = Dataset.from_pandas(
    test_encoded_swapped,
    preserve_index=False
)
print(test_dataset_swapped)

test_dataset_empty = Dataset.from_pandas(
    test_encoded_empty,
    preserve_index=False
)
print(test_dataset_empty)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 438
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 438
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 438
})


Preprocessing with shorter contexts (256 tokens)

In [32]:
# truncating text that exceeds 256 tokens

# truncating from the left (beginning)
tokenizer.truncation_side = "left"

def preprocess_short(context, question):
    return tokenizer(
        context,
        question,
        truncation="only_first", # truncates only context (from the left)
        max_length=256
    )

In [33]:
# tokenizing training data

train_tokenized_short = train.apply(
    lambda row: preprocess_short(row["preceding_turns"], row["question"]),
    axis=1
)

# converting tokenized data into dataframe
train_encoded_short = pd.DataFrame(train_tokenized_short.tolist())

# adding integer labels
train_encoded_short["labels"] = train["label_int"].to_numpy()

train_encoded_short.head()

,input_ids,attention_mask,labels
0,"[0, 152, 74, 28, 10, 7404, 7, 120, 8910, 2194,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
1,"[0, 41761, 1215, 250, 35, 20560, 203, 99, 24, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
2,"[0, 41761, 1215, 250, 35, 178, 141, 171, 82, 7...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
3,"[0, 41761, 1215, 250, 35, 20, 132, 207, 9061, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
4,"[0, 41761, 1215, 250, 35, 345, 16, 117, 3050, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0


In [34]:
# converting to Hugging Face dataset

train_dataset_short = Dataset.from_pandas(
    train_encoded_short,
    preserve_index=False
)

print(train_dataset_short)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 345
})


In [35]:
# tokenizing validation data

valid_tokenized_short = valid.apply(
    lambda row: preprocess_short(row["preceding_turns"], row["question"]),
    axis=1
)

# converting tokenized data into dataframe
valid_encoded_short = pd.DataFrame(valid_tokenized_short.tolist())

# adding integer labels
valid_encoded_short["labels"] = valid["label_int"].to_numpy()

valid_encoded_short.head()

,input_ids,attention_mask,labels
0,"[0, 59, 65, 9, 5, 10017, 142, 24, 18, 41, 1189...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0
1,"[0, 41761, 1215, 250, 35, 646, 5593, 12677, 74...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
2,"[0, 41761, 1215, 250, 35, 280, 2682, 16, 4968,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
3,"[0, 41761, 1215, 250, 35, 359, 19377, 131, 100...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1
4,"[0, 2466, 207, 15, 984, 24133, 330, 7, 984, 13...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1


In [36]:
# converting to Hugging Face dataset

valid_dataset_short = Dataset.from_pandas(
    valid_encoded_short,
    preserve_index=False
)

print(valid_dataset_short)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 90
})


### Training

In [18]:
# implementing dynamic padding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [12]:
# defining evaluation function to compute macro F1

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "macro_f1": f1_score(labels, predictions, average="macro")
    }

In [13]:
# training model simple default model

'''set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base", num_labels=2, id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir="../model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(trainer.args.device)

trainer.train()'''

'set_seed(42)\n\nmodel = AutoModelForSequenceClassification.from_pretrained(\n    "FacebookAI/roberta-base", num_labels=2, id2label=id2label, label2id=label2id\n)\n\ntraining_args = TrainingArguments(\n    output_dir="../model",\n    learning_rate=2e-5,\n    per_device_train_batch_size=16,\n    per_device_eval_batch_size=16,\n    num_train_epochs=3,\n    weight_decay=0.01,\n    eval_strategy="epoch",\n    save_strategy="epoch",\n    logging_strategy="epoch",\n    metric_for_best_model="macro_f1",\n    greater_is_better=True,\n    load_best_model_at_end=True,\n    seed=42,\n    data_seed=42,\n)\n\ntrainer = Trainer(\n    model=model,\n    args=training_args,\n    train_dataset=train_dataset,\n    eval_dataset=valid_dataset,\n    processing_class=tokenizer,\n    data_collator=data_collator,\n    compute_metrics=compute_metrics,\n)\n\nprint(trainer.args.device)\n\ntrainer.train()'

 First training attempt; macro F1 did not improve at all  
 results suggest model is predicting every question as RQ
 
 [66/66 00:50, Epoch 3/3]  
Epoch	Training Loss	Validation Loss	Macro F1  
1	No log	0.653401	0.391892  
2	No log	0.653259	0.391892  
3	No log	0.649935	0.391892  

In [14]:
'''predictions = trainer.predict(valid_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

print(np.unique(y_pred, return_counts=True))'''

'predictions = trainer.predict(valid_dataset)\ny_pred = np.argmax(predictions.predictions, axis=-1)\n\nprint(np.unique(y_pred, return_counts=True))'

confirmed: all instances predicted as RQs  

- class imbalance
- small dataset
- too few epochs
- long contexts


In [15]:
# trying more epochs

# training model

'''set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base", num_labels=2, id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir="../model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(trainer.args.device)

trainer.train()'''

'set_seed(42)\n\nmodel = AutoModelForSequenceClassification.from_pretrained(\n    "FacebookAI/roberta-base", num_labels=2, id2label=id2label, label2id=label2id\n)\n\ntraining_args = TrainingArguments(\n    output_dir="../model",\n    learning_rate=2e-5,\n    per_device_train_batch_size=16,\n    per_device_eval_batch_size=16,\n    num_train_epochs=6,\n    weight_decay=0.01,\n    eval_strategy="epoch",\n    save_strategy="epoch",\n    logging_strategy="epoch",\n    metric_for_best_model="macro_f1",\n    greater_is_better=True,\n    load_best_model_at_end=True,\n    seed=42,\n    data_seed=42,\n)\n\ntrainer = Trainer(\n    model=model,\n    args=training_args,\n    train_dataset=train_dataset,\n    eval_dataset=valid_dataset,\n    processing_class=tokenizer,\n    data_collator=data_collator,\n    compute_metrics=compute_metrics,\n)\n\nprint(trainer.args.device)\n\ntrainer.train()'

- training loss decreases slightly (0.686 to 0.625)
- validation loss barely decreases + increases again during 6th epoch
- macro F1 at 0.3919 for all epochs (still seems to predict majority class)

possible reasons:  
- class imbalance
- small training data
- long context + short questions

second attempt:  

 [132/132 01:39, Epoch 6/6]  
Epoch	Training Loss	Validation Loss	Macro F1  
1	0.688295	0.654703	0.391892  
2	0.677425	0.653386	0.391892  
3	0.666970	0.648406	0.391892  
4	0.647801	0.643337	0.391892  
5	0.622557	0.637140	0.387755  
6	0.604873	0.635651	0.419823  

- macro F1 changes during epoch 5 and 6
- both training and validation loss decrease (even by epoch 6)


third attempt (with seed):  

 [132/132 01:38, Epoch 6/6]  
Epoch	Training Loss	Validation Loss	Macro F1  
1	0.707848	0.648813	0.391892  
2	0.693848	0.657416	0.391892  
3	0.671956	0.648383	0.391892  
4	0.644129	0.647895	0.387755  
5	0.642201	0.644701	0.387755  
6	0.634771	0.644807	0.387755  

- training loss decreases until epoch 6
- validation loss increases slightly at epoch 6
- F1 is the same for epochs 1 to 3, and epochs 4 to 6  


In [16]:
# inspecting rhetorical probabilities to see if the model is learning

'''predictions = trainer.predict(valid_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

probs = torch.softmax(
    torch.tensor(predictions.predictions), dim=1
).numpy()

pd.Series(probs[:, 1]).describe()'''

'predictions = trainer.predict(valid_dataset)\ny_pred = np.argmax(predictions.predictions, axis=-1)\n\nprobs = torch.softmax(\n    torch.tensor(predictions.predictions), dim=1\n).numpy()\n\npd.Series(probs[:, 1]).describe()'

In [17]:
#print(confusion_matrix(valid["label_int"], y_pred))

Trying balanced classes (since this improved baseline performance as well)

In [18]:
# trying class-weighted cross-entropy loss

# computing class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train["label_int"]
)

print(class_weights)

[1.28731343 0.81753555]


In [19]:
# defining a costum Trainer

class WeightedTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels") # extracting gold labels from batch
        outputs = model(**inputs)

        loss_fn = nn.CrossEntropyLoss(
            weight=torch.tensor(
                class_weights,
                dtype=torch.float32,
                device=outputs.logits.device
            )
        )

        loss = loss_fn(outputs.logits, labels)

        return (loss, outputs) if return_outputs else loss

In [20]:
# training new model

set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base", num_labels=2, id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir="../model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=42,
    data_seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(trainer.args.device)

trainer.train()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11970.53it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda:0


Epoch,Training Loss,Validation Loss,Macro F1
1,0.713132,0.690667,0.419823
2,0.689920,0.694452,0.304029
3,0.692554,0.689278,0.513221
4,0.667573,0.687181,0.512614
5,0.649353,0.686526,0.524887
6,0.625654,0.684447,0.560185


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]


TrainOutput(global_step=132, training_loss=0.673031048341231, metrics={'train_runtime': 95.656, 'train_samples_per_second': 21.64, 'train_steps_per_second': 1.38, 'total_flos': 544639884595200.0, 'train_loss': 0.673031048341231, 'epoch': 6.0})

 [132/132 01:37, Epoch 6/6]  
Epoch	Training Loss	Validation Loss	Macro F1  
1	0.712683	0.690992	0.419823  
2	0.690267	0.695115	0.300000  
3	0.689238	0.690552	0.483324  
4	0.669704	0.687788	0.506355  
5	0.654828	0.687602	0.534737  
6	0.640948	0.685624	0.537037  

- improvement compared to non-balanced weights


In [21]:
# checking confusion matrix

predictions = trainer.predict(valid_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

print(confusion_matrix(valid["label_int"], y_pred))

[[17 15]
 [23 35]]


In [22]:
# saving model

trainer.save_model("../model/roberta_weighted")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.08it/s]


Trying more epochs (10)

In [ ]:
'''set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="../model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=42,
    data_seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(trainer.args.device)

trainer.train()'''

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11635.09it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda:0


Epoch,Training Loss,Validation Loss,Macro F1
1,0.710490,0.690126,0.387755
2,0.694266,0.697375,0.262295
3,0.694281,0.690205,0.532468
4,0.659868,0.690313,0.550945
5,0.617869,0.713955,0.567552
6,0.580129,0.748163,0.555844
7,0.445351,0.900318,0.477713
8,0.333809,0.986009,0.489346
9,0.250050,1.154357,0.466650
10,0.205403,1.232957,0.453870


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]


TrainOutput(global_step=220, training_loss=0.5191515965895219, metrics={'train_runtime': 159.5288, 'train_samples_per_second': 21.626, 'train_steps_per_second': 1.379, 'total_flos': 907165807778880.0, 'train_loss': 0.5191515965895219, 'epoch': 10.0})

 [220/220 02:40, Epoch 10/10]  
Epoch	Training Loss	Validation Loss	Macro F1  
1	0.710490	0.690126	0.387755  
2	0.694266	0.697375	0.262295  
3	0.694281	0.690205	0.532468  
4	0.659868	0.690313	0.550945  
5	0.617869	0.713955	0.567552  
6	0.580129	0.748163	0.555844  
7	0.445351	0.900318	0.477713  
8	0.333809	0.986009	0.489346  
9	0.250050	1.154357	0.466650  
10	0.205403	1.232957	0.453870  

- overfitting!!
- best F1 in epoch 5

In [ ]:
# checking confusion matrix

'''predictions = trainer.predict(valid_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

print(confusion_matrix(valid["label_int"], y_pred))'''

[[12 20]
 [14 44]]


In [ ]:
# checking that it saved the best checkpoint

'''print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)'''

../model/checkpoint-110
0.567552289429056


In [28]:
# saving best checkpoint from 10 epoch model

trainer.save_model("../model/roberta_weighted_10_epochs")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.05s/it]


Trying increasing learning rate (3e-5)

In [ ]:
'''set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="../model",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=42,
    data_seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(trainer.args.device)

trainer.train()'''

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11453.49it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda:0


Epoch,Training Loss,Validation Loss,Macro F1
1,0.713886,0.689677,0.391892
2,0.696732,0.699538,0.262295
3,0.700025,0.692283,0.537536
4,0.683748,0.690779,0.494712
5,0.683300,0.689513,0.544534
6,0.670988,0.686421,0.518214


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.06it/s]


TrainOutput(global_step=132, training_loss=0.6914465210654519, metrics={'train_runtime': 95.5997, 'train_samples_per_second': 21.653, 'train_steps_per_second': 1.381, 'total_flos': 544639884595200.0, 'train_loss': 0.6914465210654519, 'epoch': 6.0})

 [132/132 01:36, Epoch 6/6]  
Epoch	Training Loss	Validation Loss	Macro F1  
1	0.713886	0.689677	0.391892  
2	0.696732	0.699538	0.262295  
3	0.700025	0.692283	0.537536  
4	0.683748	0.690779	0.494712  
5	0.683300	0.689513	0.544534  
6	0.670988	0.686421	0.518214  

- results not better than with lower learning rate
- slightly better than 6 epochs and 2e-5
- but not better than best checkpoint from 10 epochs and 2e-5

In [30]:
predictions = trainer.predict(valid_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

print(confusion_matrix(valid["label_int"], y_pred))

[[18 14]
 [26 32]]


In [31]:
# saving best checkpoint from 3e-5 model

trainer.save_model("../model/roberta_weighted_higher_lr")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]


Trying shorter contexts (256 tokens)

In [ ]:
'''set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(
    "FacebookAI/roberta-base",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="../model",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=42,
    data_seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_short,
    eval_dataset=valid_dataset_short,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(trainer.args.device)

trainer.train()'''

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11992.07it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda:0


Epoch,Training Loss,Validation Loss,Macro F1
1,0.707422,0.689424,0.391892
2,0.692670,0.692931,0.457014
3,0.687719,0.699487,0.400705
4,0.675043,0.682690,0.585500
5,0.622168,0.710987,0.495156
6,0.530628,0.750025,0.532468


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.06it/s]


TrainOutput(global_step=132, training_loss=0.6526084451964407, metrics={'train_runtime': 56.8976, 'train_samples_per_second': 36.381, 'train_steps_per_second': 2.32, 'total_flos': 272319942297600.0, 'train_loss': 0.6526084451964407, 'epoch': 6.0})

 [132/132 00:57, Epoch 6/6]  
Epoch	Training Loss	Validation Loss	Macro F1  
1	0.707422	0.689424	0.391892  
2	0.692670	0.692931	0.457014  
3	0.687719	0.699487	0.400705  
4	0.675043	0.682690	0.585500  
5	0.622168	0.710987	0.495156  
6	0.530628	0.750025	0.532468  

- best F1 result so far in epoch 4: 0.5855
- signs of overfitting after epoch 4

In [38]:
predictions = trainer.predict(valid_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

print(confusion_matrix(valid["label_int"], y_pred))

[[11 21]
 [ 9 49]]


In [39]:
# saving best checkpoint from 256 token model

trainer.save_model("../model/roberta_weighted_short")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.03it/s]


In [40]:
predictions = trainer.predict(valid_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)

print(f1_score(valid["label_int"], y_pred, average="macro"))
print(confusion_matrix(valid["label_int"], y_pred))

0.5943509615384616
[[11 21]
 [ 9 49]]


- model favors RQs (more so than the previous models)

### Overview for all experimental models

Configuration	&emsp; &emsp;    Dev macro F1	 &emsp;    IQ recall	&emsp;     RQ recall  
512 tokens,6 epochs,2e-5	 &emsp;    0.537	&emsp;     50.0%    &emsp; 	58.6%  
512 tokens, 6 epochs, 3e-5	&emsp;     0.545    &emsp; 	56.3% &emsp;   	55.2%  
256 tokens, 6 epochs, 2e-5	&emsp;     0.594	&emsp;     34.4%   &emsp;  	84.5%  


...

Final model: 10 epoch, 2e-5, 512 tokens   
Because: second best validation macro F1, context closer to logistic regression model (less truncated)

### Evaluation on the test set

In [19]:
# loading final model and creating evaluation trainer

final_model = AutoModelForSequenceClassification.from_pretrained(
    "../model/roberta_weighted_10_epochs"
)

eval_trainer = Trainer(
    model=final_model,
    processing_class=tokenizer,
    data_collator=data_collator,
)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11559.31it/s]

In [20]:
# checking that the saved model produces the same results as the best checkpoint

predictions_valid = eval_trainer.predict(valid_dataset)

y_pred_valid = np.argmax(predictions_valid.predictions, axis=-1)

print("Macro F1:", f1_score(valid_dataset["labels"], y_pred_valid, average="macro"))
print("Confusion matrix:")
print(confusion_matrix(valid_dataset["labels"], y_pred_valid))

Macro F1: 0.567552289429056
Confusion matrix:
[[12 20]
 [14 44]]


In [ ]:
# checking that all splits contain the same labels in the same order

assert (
    list(test_dataset_real["labels"])
    == list(test_dataset_swapped["labels"])
    == list(test_dataset_empty["labels"])
)

In [24]:
# real context

predictions_real = eval_trainer.predict(test_dataset_real)

y_pred_real = np.argmax(predictions_real.predictions, axis=-1)

print("Macro F1:", f1_score(test_dataset_real["labels"], y_pred_real, average="macro"))
print("Confusion matrix:")
print(confusion_matrix(test_dataset_real["labels"], y_pred_real))

Macro F1: 0.5439903665236697
Confusion matrix:
[[ 46  94]
 [ 72 226]]


In [25]:
# swapped context

predictions_swapped = eval_trainer.predict(test_dataset_swapped)

y_pred_swapped = np.argmax(predictions_swapped.predictions, axis=-1)

print("Macro F1:", f1_score(test_dataset_swapped["labels"], y_pred_swapped, average="macro"))
print("Confusion matrix:")
print(confusion_matrix(test_dataset_swapped["labels"], y_pred_swapped))

Macro F1: 0.5061022334361275
Confusion matrix:
[[ 40 100]
 [ 81 217]]


In [26]:
# empty context

predictions_empty = eval_trainer.predict(test_dataset_empty)

y_pred_empty = np.argmax(predictions_empty.predictions, axis=-1)

print("Macro F1:", f1_score(test_dataset_empty["labels"], y_pred_empty, average="macro"))
print("Confusion matrix:")
print(confusion_matrix(test_dataset_empty["labels"], y_pred_empty))

Macro F1: 0.25714747213697303
Confusion matrix:
[[140   0]
 [294   4]]


In [29]:
# adding individual predctions to copy of test df

results = test.copy()

results["pred_real"] = y_pred_real
results["pred_swapped"] = y_pred_swapped
results["pred_empty"] = y_pred_empty

results

,author,question_ending,question,preceding_turns,paragraph,paragraph_with_author_and_marker,full_turn,full_turn_with_marker,following_turns,subreddit,label,context_length,length_group,swapped_context,label_int,pred_real,pred_swapped,pred_empty
0,Person_[deleted],so,how so?,"Person_A: It's balkaninsight, of course the te...",Person_[deleted]: how so?,Person_[deleted]: <ques>how so?</ques>,Person_[deleted]: how so?,Person_[deleted]: <ques>how so?</ques>,Person_B: > 1) I don't think Europeans get rem...,IAmA,informational,37,0,Person_A: That's not really true. Multiple com...,0,0,1,0
1,Person_A,why not,Doctors also look up things (in front of their...,Person_A: So obviously a smart kid with Intern...,1.) Doctors also look up things (in front of t...,Person_A: 1.) <ques>Doctors also look up thing...,"Person_A: False dichotomy, both because \n\n1....","Person_A: False dichotomy, both because \n\n1....",Person_B: > &gt; Doctors also look up things (...,changemyview,rhetorical,529,3,Person_[deleted]: so I did a search and read t...,1,1,0,1
2,Person_B,so,So?,Person_A: &gt; Doesn't Russia consider Crimea ...,So? There's no way in which Russia could mobil...,Person_B: <ques>So?</ques> There's no way in w...,Person_B: &gt; Right. Except when it isn't. \...,Person_B: &gt; Right. Except when it isn't. \...,Person_A: > You're putting words in my mouth t...,europe,rhetorical,1149,3,Person_A: One big argument against communism i...,1,1,1,0
3,Person_A,what,"Do the police just disappear them secretly, or...",Person_[deleted]: so I did a search and read t...,Person_A: Which gives you exactly the problems...,Person_A: Which gives you exactly the problems...,Person_A: Which gives you exactly the problems...,Person_A: Which gives you exactly the problems...,Person_B: > &gt;Do the police just disappear t...,changemyview,rhetorical,685,3,"Person_A: By definition, a matriarchy would ca...",1,0,1,0
4,Person_B,how,Which makes you different than American fanboy...,"Person_A: Either intervene in every case, whet...","Hmm, sounds like you base which rebels to supp...","Person_B: Hmm, sounds like you base which rebe...","Person_B: &gt;are islamists who, at least in t...","Person_B: &gt;are islamists who, at least in t...",,europe,rhetorical,444,3,Person_A: &gt; How is the current system biase...,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433,Person_[deleted],how,How?,Person_A: Morality...good vs. evil...right vs....,&gt; In the presence of intense scrutiny by lo...,Person_[deleted]: &gt; In the presence of inte...,Person_[deleted]: &gt; In the presence of inte...,Person_[deleted]: &gt; In the presence of inte...,Person_A: > Abortion is the termination of a l...,changemyview,informational,219,1,Person_A: Going to try to throw in a special c...,0,1,1,0
434,Person_B,what,"Are you retarded, or what?",Person_A: We're doing a better job fighting ag...,"Person_B: Are you retarded, or what? Schengen ...","Person_B: <ques>Are you retarded, or what?</qu...","Person_B: Are you retarded, or what? Schengen ...","Person_B: <ques>Are you retarded, or what?</qu...","Person_A: > No, you're the retarded one. Being...",europe,rhetorical,212,1,Person_A: It's impossible to make a blanket st...,1,1,1,0
435,Person_A,why,Why?,Person_A: <non-textual content>\nPerson_B: Wha...,Person_A: You are the country that has been se...,Person_A: You are the country that has been se...,Person_A: You are the country that has been se...,Person_A: You are the country that has been se...,Person_C: > Where do you live if it's not a se...,europe,rhetorical,12,0,Person_A: The memes have gone to far.\nPerson_...,1,0,0,0
436,Person_B,how,How?,Person_A: Is it really that lucrative for Alba...,&gt; And how do these people still recieve the...,Person_B: &gt; And how do these people still r...,"Person_B: Oh, that was some other thread. Sorr...","Person_B: Oh, that was some other thread. Sorr...",,europe,rhetorical,239,2,Person_A: https://en.wikipedia.org/wiki/Ad_hom

In [31]:
# prediction change rates (sensitivity to manipulation)

(y_pred_real != y_pred_swapped).mean()

np.float64(0.3356164383561644)